# Checkpoint 2: Enhanced Model Development

Checkpoint 2 of the CSE 572 Data Mining project. We will:
1.  Load the data and define our metric functions (from Checkpoint 1).
2.  Create an enhanced feature set (V2) with price and categorical features.
3.  Tune the LightGBM model using Optuna.
4.  Explore a new model (Random Forest) for comparison.
5.  Generate a final results table comparing all models.

## 1. Setup and Data Loading

Import libraries, load the three source CSVs, and apply memory reduction.

In [44]:
print('hello2')

hello2


In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
print("Hello2")
DATA_DIR = Path('/Users/AfshiyaRoohi/Downloads/DM/content')
sales_path = DATA_DIR / 'sales_train_validation.csv'
calendar_path = DATA_DIR / 'calendar.csv'
prices_path = DATA_DIR / 'sell_prices.csv'

#
def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to reduce memory usage."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type)[:3] != 'dat':
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min >= 0:
                    if c_max < 255:
                        df[col] = df[col].astype(np.uint8)
                    elif c_max < 65535:
                        df[col] = df[col].astype(np.uint16)
                    elif c_max < 4294967295:
                        df[col] = df[col].astype(np.uint32)
                    else:
                        df[col] = df[col].astype(np.uint64)
                else:
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                    else:
                        df[col] = df[col].astype(np.int64)
            else:
                df[col] = pd.to_numeric(df[col], downcast='float')
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Memory usage reduced from {start_mem:.2f} MB to {end_mem:.2f} MB")
    return df

print("Hello")
SAMPLE_SIZE = None
sales_df_full = pd.read_csv(sales_path)
print(f"Original shape of sales_df: {sales_df_full.shape}")
sales_df =  sales_df_full# sales_df_full.sample(SAMPLE_SIZE, random_state=42)
print(f"New sampled shape of sales_df: {sales_df.shape}")
print(sales_df_full)
del sales_df_full
gc.collect()

sales_df = reduce_mem_usage(sales_df)
calendar_df = reduce_mem_usage(pd.read_csv(calendar_path))
prices_df = reduce_mem_usage(pd.read_csv(prices_path))


Hello2
Hello
Original shape of sales_df: (30490, 1919)
New sampled shape of sales_df: (30490, 1919)
                                  id        item_id    dept_id   cat_id  \
0      HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES   
1      HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES   
2      HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES   
3      HOBBIES_1_004_CA_1_validation  HOBBIES_1_004  HOBBIES_1  HOBBIES   
4      HOBBIES_1_005_CA_1_validation  HOBBIES_1_005  HOBBIES_1  HOBBIES   
...                              ...            ...        ...      ...   
30485    FOODS_3_823_WI_3_validation    FOODS_3_823    FOODS_3    FOODS   
30486    FOODS_3_824_WI_3_validation    FOODS_3_824    FOODS_3    FOODS   
30487    FOODS_3_825_WI_3_validation    FOODS_3_825    FOODS_3    FOODS   
30488    FOODS_3_826_WI_3_validation    FOODS_3_826    FOODS_3    FOODS   
30489    FOODS_3_827_WI_3_validation    FOODS_3_827    FOODS_3    FOODS   


## 2. Merge Datasets

Melt the sales data into a long format and merge with calendar and price data.

In [46]:
df_long = sales_df.melt(
    id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name='d',
    value_name='sales'
)

merged_df = df_long.merge(calendar_df, how='left', left_on='d', right_on='d')
merged_df = merged_df.merge(prices_df, how='left', on=['store_id', 'item_id', 'wm_yr_wk'])
merged_df['date'] = pd.to_datetime(merged_df['date'])

del df_long, sales_df, calendar_df, prices_df
gc.collect()

print(f"Merged dataset shape: {merged_df.shape}")
merged_df.head()

Merged dataset shape: (58327370, 22)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN


## 3. Metric Definitions

Define the MAE, sMAPE, and WRMSSE functions from Checkpoint 1 for evaluation.

In [47]:
from typing import Tuple

# Mean Absolute Error
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Symmetric Mean Absolute Percentage Error (sMAPE)
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred))
    diff = np.abs(y_true - y_pred)
    mask = denominator != 0
    smape = 100 * np.mean(2.0 * diff[mask] / denominator[mask])
    return smape

# === WRMSSE helpers (robust) ===
def _check_shapes(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        raise ValueError(f"y_true {y_true.shape} and y_pred {y_pred.shape} must match")
    if y_true.ndim != 2:
        raise ValueError("Expected 2D arrays: (n_series, horizon)")

def ids_in(df):
    return set(df['id'].unique())

def compute_scales(train_y, seasonality=1, eps=1e-8):
    if seasonality < 1:
        seasonality = 1
    diffs = train_y[:, seasonality:] - train_y[:, :-seasonality]
    scale = np.sqrt(np.mean(diffs**2, axis=1))
    scale = np.where(scale <= 0.0, eps, scale)
    return scale

def compute_weights(train_y, price=None, normalize=True, eps=1e-8):
    mean_qty = train_y.mean(axis=1)
    w = mean_qty if price is None else (mean_qty * np.asarray(price))
    w = np.where(w < 0, 0, w) + eps
    if normalize:
        w = w / w.sum()
    return w

def wrmsse(y_true, y_pred, scales, weights):
    _check_shapes(y_true, y_pred)
    if len(scales)  != y_true.shape[0]: raise ValueError("len(scales) != n_series")
    if len(weights) != y_true.shape[0]: raise ValueError("len(weights) != n_series")
    series_rmse  = np.sqrt(np.mean((y_true - y_pred)**2, axis=1))
    series_rmsse = series_rmse / scales
    valid = np.isfinite(series_rmsse) & np.isfinite(weights)
    if not np.any(valid):
        return np.nan
    w_norm = weights[valid] / weights[valid].sum()
    return float(np.sum(w_norm * series_rmsse[valid]))

In [48]:
# --- NEW CELL: Create Global Metrics & True Values ---

def create_lag_features(df, group_cols, lag_list):
    """Creates lag features for the sales column grouped by id."""
    for lag in lag_list:
        df[f'lag_{lag}'] = df.groupby(group_cols)['sales'].shift(lag)
    return df

def create_roll_mean_features(df, group_cols, windows):
    """Creates rolling mean features for the sales column grouped by id."""
    for window in windows:
        # We shift by 1 to ensure the rolling mean is calculated on past data only
        df[f'rolling_mean_{window}'] = df.groupby(group_cols)['sales'].shift(1).rolling(window).mean().reset_index(0,drop=True)
    return df

print("Creating global metrics from full history...")
# Define the global split date
# (d_1 to d_1913 is train, d_1914 to d_1941 is valid)
date_max = merged_df['date'].max()
forecast_horizon = 28
validation_start_date = date_max - pd.Timedelta(days=forecast_horizon-1)

# 1. Create full train/valid DFs (NO dropna)
train_df_full_history = merged_df[merged_df['date'] < validation_start_date]
valid_df_full_history = merged_df[merged_df['date'] >= validation_start_date]

# 2. Get the complete list of series IDs from the full history
# These IDs must be used for all matrix building
GLOBAL_SERIES_IDS = sorted(list(train_df_full_history['id'].unique()))

# 3. Build the matrices needed for metric calculation
def build_matrix(df, ids, value_col):
    # Ensure all series are present, fill missing with 0
    df_pivoted = df.pivot(index='id', columns='date', values=value_col).reindex(ids)
    return df_pivoted.to_numpy(na_value=0.0) # Fill NaNs with 0

print(f"Building matrices for {len(GLOBAL_SERIES_IDS)} total series...")
# Build from the FULL training history (no dropna)
train_y_full_matrix = build_matrix(train_df_full_history, GLOBAL_SERIES_IDS, 'sales')

# Build the ground truth matrix (d_1914 to d_1941)
# This will be used to evaluate ALL models
TRUE_MATRIX = build_matrix(valid_df_full_history, GLOBAL_SERIES_IDS, 'sales')

# 4. Calculate scales and weights ONCE from the full history
GLOBAL_SCALES = compute_scales(train_y_full_matrix, seasonality=7)
GLOBAL_WEIGHTS = compute_weights(train_y_full_matrix, price=None, normalize=True)

print("Global metrics (scales, weights, true_matrix) created.")

# Clean up to save memory
del train_df_full_history, valid_df_full_history, train_y_full_matrix
gc.collect()

Creating global metrics from full history...
Building matrices for 30490 total series...
Global metrics (scales, weights, true_matrix) created.


0

## 4. Enhanced Feature Engineering (V2)

Create our V2 dataset, which includes price and categorical features as planned in Checkpoint 1.

In [49]:
# --- This is the updated Cell 4 ---
print("Starting Checkpoint 2 Feature Engineering (V2)...")

features_df_v2 = merged_df.copy()

# --- Create Basic Calendar Features ---
features_df_v2['day_of_week'] = features_df_v2['date'].dt.weekday
features_df_v2['week_of_year'] = features_df_v2['date'].dt.isocalendar().week.astype(int)
features_df_v2['month'] = features_df_v2['date'].dt.month
features_df_v2['year'] = features_df_v2['date'].dt.year

# --- Create Lag & Rolling Features ---
# This will create NaNs for the first 28 days, which is correct.
features_df_v2 = create_lag_features(features_df_v2, group_cols=['id'], lag_list=[1, 7, 14, 28])
features_df_v2 = create_roll_mean_features(features_df_v2, group_cols=['id'], windows=[7, 28])

# --- Create Price Features ---
features_df_v2['price_lag_1'] = features_df_v2.groupby(['id'])['sell_price'].shift(1)
features_df_v2['price_change_1'] = features_df_v2['sell_price'] / features_df_v2['price_lag_1'] - 1.0
features_df_v2['price_change_7'] = features_df_v2['sell_price'] / features_df_v2.groupby(['id'])['sell_price'].shift(7) - 1.0

# --- Handle Categorical Features ---
event_cols = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
for col in event_cols:
    features_df_v2[col] = features_df_v2[col].fillna('None')

categorical_features = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1', 'event_type_1']
for col in categorical_features:
    if col in features_df_v2.columns:
        features_df_v2[col] = features_df_v2[col].astype('category')

# --- Define Feature List ---
features_v2 = [
    'day_of_week','week_of_year','month','year',
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28',
    'sell_price', 'price_lag_1', 'price_change_1', 'price_change_7',
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id',
    'event_name_1', 'event_type_1'
]
target_col = 'sales'

# --- !! CRITICAL CHANGE !! ---
# --- Split FIRST, then drop NaNs from the training set ONLY ---

print("Splitting data based on time...")
date_max = features_df_v2['date'].max()
forecast_horizon = 28
validation_start_date = date_max - pd.Timedelta(days=forecast_horizon - 1)

# 1. Create the train set (full history, d_1 to d_1913)
train_df_v2 = features_df_v2[features_df_v2['date'] < validation_start_date].copy()

# 2. Create the validation set (d_1914 to d_1941)
valid_df_v2 = features_df_v2[features_df_v2['date'] >= validation_start_date].copy()

# 3. Create the data for calculating metrics (needs full history)
# We will use 'train_df_v2' and 'valid_df_v2' for this in the metric cell.

# 4. Now, create the X/y for *training* by dropping NaNs *only from the training set*
X_train_v2 = train_df_v2.dropna(subset=features_v2)[features_v2]
y_train_v2 = train_df_v2.dropna(subset=features_v2)[target_col]

# X_valid_v2 will have NaNs for lags (e.g., lag_1 on day 1914), but LGBM can handle them.
X_valid_v2 = valid_df_v2[features_v2]
y_valid_v2 = valid_df_v2[target_col] # This is just the y_true for MAE/sMAPE
y_valid_df_v2_metrics = valid_df_v2[['id', 'date', 'sales']].copy() # For WRMSSE calculation

print(f"V2 Training features shape (after dropna): {X_train_v2.shape}")
print(f"V2 Validation features shape: {X_valid_v2.shape}")

Starting Checkpoint 2 Feature Engineering (V2)...
Splitting data based on time...
V2 Training features shape (after dropna): (44712825, 21)
V2 Validation features shape: (853720, 21)


## 5. Hyperparameter Tuning (Optuna)

We will use Optuna to find the best hyperparameters for our LightGBM model.

In [50]:
!pip install optuna optuna_integration
import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb # Make sure lgb is imported

def objective(trial):
    """Define the objective function for Optuna to optimize."""

    params = {
        'objective': 'tweedie',
        'metric': 'mae',
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'n_jobs': -1,
        'random_state': 42
    }

    # Note: We pass categorical_feature to .fit(), not the constructor
    model = LGBMRegressor(**params)

    model.fit(X_train_v2, y_train_v2,
              eval_set=[(X_valid_v2, y_valid_v2)],
              eval_metric='mae',
              callbacks=[optuna.integration.LightGBMPruningCallback(trial, 'l1'),
                         lgb.early_stopping(50, verbose=False)],
              categorical_feature=categorical_features
             )

    preds = model.predict(X_valid_v2)
    mae_score = mean_absolute_error(y_valid_v2, preds)
    return mae_score

# Run the Optuna study
print("Starting Optuna study...")
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=5) # Using 30 trials for speed. Increase meeku time unte

print("Study finished!")
print(f"Best MAE: {study.best_value}")
print("Best parameters:")
print(study.best_params)

# Store the best parameters
final_lgb_params = study.best_params

[I 2025-12-03 05:14:30,877] A new study created in memory with name: no-name-937b4d99-5d74-4913-bf4b-ae16fdb2aa76


Starting Optuna study...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.786190 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4740
[LightGBM] [Info] Number of data points in the train set: 44712825, number of used features: 21
[LightGBM] [Info] Start training from score 0.353637


[I 2025-12-03 05:24:14,787] Trial 0 finished with value: 0.9892402465781058 and parameters: {'learning_rate': 0.06299627278938795, 'num_leaves': 61, 'max_depth': 17, 'subsample': 0.831780906470466, 'colsample_bytree': 0.7525282366239546}. Best is trial 0 with value: 0.9892402465781058.


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.364118 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4740
[LightGBM] [Info] Number of data points in the train set: 44712825, number of used features: 21
[LightGBM] [Info] Start training from score 0.353637


[I 2025-12-03 05:32:18,668] Trial 1 finished with value: 0.9921171124725601 and parameters: {'learning_rate': 0.05866652501100579, 'num_leaves': 78, 'max_depth': 7, 'subsample': 0.7446120228092126, 'colsample_bytree': 0.9805355855300784}. Best is trial 0 with value: 0.9892402465781058.


...

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.852989 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4740
[LightGBM] [Info] Number of data points in the train set: 44712825, number of used features: 21
[LightGBM] [Info] Start training from score 0.353637
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[I 2025-12-03 05:47:44,873] Trial 3 finished with value: 0.9981466656471232 and parameters: {'learning_rate': 0.04923299925059431, 'num_leaves': 70, 'max_depth': 6, 'subsample': 0.6515811705668569, 'colsample_bytree': 0.8997579184118552}. Best is trial 0 with value: 0.9892402465781058.


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.944156 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4740
[LightGBM] [Info] Number of data points in the train set: 44712825, number of used features: 21
[LightGBM] [Info] Start training from score 0.353637


[I 2025-12-03 05:57:01,288] Trial 4 finished with value: 0.9872338310402305 and parameters: {'learning_rate': 0.06794580482636105, 'num_leaves': 62, 'max_depth': 17, 'subsample': 0.6812421538694611, 'colsample_bytree': 0.8670698615463088}. Best is trial 4 with value: 0.9872338310402305.


Study finished!
Best MAE: 0.9872338310402305
Best parameters:
{'learning_rate': 0.06794580482636105, 'num_leaves': 62, 'max_depth': 17, 'subsample': 0.6812421538694611, 'colsample_bytree': 0.8670698615463088}


In [51]:
# print("Optuna results loaded from your saved parameters.")
# final_lgb_params = {
#     'learning_rate': 0.09788369711168385,
#     'num_leaves': 89,
#     'max_depth': 13,
#     'subsample': 0.6845876314605502,
#     'colsample_bytree': 0.9999624618551161
# }
# print(f"Loaded parameters: {final_lgb_params}")

## 6. Training the Tuned LightGBM Model

Using the best parameters found by Optuna, we will now train our final, "improved" LightGBM model.

In [52]:
print("Training final Tuned LightGBM model...")

final_lgb_params['n_estimators'] = 1500
final_lgb_params['random_state'] = 42
final_lgb_params['n_jobs'] = -1
final_lgb_params['objective'] = 'tweedie'
final_lgb_params['metric'] = 'mae'

lgb_model_tuned = LGBMRegressor(**final_lgb_params)

lgb_model_tuned.fit(X_train_v2, y_train_v2,
                    eval_set=[(X_valid_v2, y_valid_v2)],
                    eval_metric='mae',
                    callbacks=[lgb.early_stopping(100, verbose=100)],
                    categorical_feature=categorical_features
                   )

lgb_preds_tuned = lgb_model_tuned.predict(X_valid_v2)

lgb_tuned_df = valid_df_v2[['id', 'date']].copy()
lgb_tuned_df['pred'] = lgb_preds_tuned
print("Tuned LightGBM predictions created.")

Training final Tuned LightGBM model...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.918039 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4740
[LightGBM] [Info] Number of data points in the train set: 44712825, number of used features: 21
[LightGBM] [Info] Start training from score 0.353637
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1500]	valid_0's l1: 0.984313
Tuned LightGBM predictions created.


## 7. Exploring a New Method: Random Forest

We will now "explore a variety of predictive methods" by training a Random Forest Regressor on the same V2 feature set.

In [55]:
print("hello")

hello


In [54]:
from sklearn.ensemble import RandomForestRegressor

print("Training FAST Random Forest model...")

# RF doesn't support category dtype directly, so we must convert them to codes.
# We also drop event text columns as RF can't handle them.
rf_feature_cols = [col for col in features_v2 if col not in ['event_name_1', 'event_type_1']]

X_train_rf = X_train_v2[rf_feature_cols].copy()
X_valid_rf = X_valid_v2[rf_feature_cols].copy()

for col in X_train_rf.select_dtypes(include=['category']).columns:
    X_train_rf[col] = X_train_rf[col].cat.codes
    X_valid_rf[col] = X_valid_rf[col].cat.codes

rf_model = RandomForestRegressor(
    n_estimators=30,      # Reduced from 100 to 30 (much faster)
    max_depth=12,         # Reduced from 15
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=100  # Increased from 10 (less detailed, faster)
)

rf_model.fit(X_train_rf, y_train_v2)

rf_preds = rf_model.predict(X_valid_rf)

rf_df = valid_df_v2[['id', 'date']].copy()
rf_df['pred'] = rf_preds
print("Random Forest predictions created.")

Training FAST Random Forest model...


KeyboardInterrupt: 

## 8. Checkpoint 2: Final Results & Analysis

Finally, we will combine our Checkpoint 1 results with our new Checkpoint 2 models to create a final comparison table.

In [ ]:
# --- 1. Load Checkpoint 1 Results (Your code is fine) ---
try:
    c1_results = pd.read_csv('checkpoint1_results.csv', index_col=0)
    print("Loaded results from checkpoint1_results.csv")
except FileNotFoundError:
    print("Warning: checkpoint1_results.csv not found. Using placeholder C1 data.")
    c1_data = {
        "Naive": {"WRMSSE": 0.9119, "MAE": 1.1699, "sMAPE": 132.11},
        "Seasonal Naive": {"WRMSSE": 0.9284, "MAE": 1.1963, "sMAPE": 133.59},
        "LightGBM (Prelim)": {"WRMSSE": 0.7154, "MAE": 1.0214, "sMAPE": 142.47}
    }
    c1_results = pd.DataFrame(c1_data).T

# --- 2. Get common series_ids and build matrices for V2 data ---
# Use the GLOBAL_SERIES_IDS defined in the new cell you added
series_ids_v2 = sorted(list(ids_in(train_df_v2) & ids_in(valid_df_v2)))

print(f"V2 Matrices built with {len(series_ids_v2)} common series.")

# This is the function you should have added in step 2.
def build_matrix(df, ids, value_col):
    df_pivoted = df.pivot(index='id', columns='date', values=value_col).reindex(ids)
    return df_pivoted.to_numpy(na_value=0.0) # Fill NaNs with 0

# Build from the FULL training history (d_1 to d_1913)
train_y_v2_matrix = build_matrix(train_df_v2, series_ids_v2, 'sales')

# Build the ground truth matrix (d_1914 to d_1941)
true_matrix_v2 = build_matrix(y_valid_df_v2_metrics, series_ids_v2, 'sales')

# Build prediction matrices
lgb_tuned_matrix = build_matrix(lgb_tuned_df, series_ids_v2, 'pred')
rf_matrix = build_matrix(rf_df, series_ids_v2, 'pred')


print(f"Shape of true values: {true_matrix_v2.shape}")
print(f"Shape of LGBM preds:  {lgb_tuned_matrix.shape}")
print(f"Shape of RF preds:    {rf_matrix.shape}")

# --- 3. Compute V2 scales & weights ---
# Calculate scales and weights using the FULL training history matrix
scales_v2  = compute_scales(train_y_v2_matrix, seasonality=7, eps=1e-8)
weights_v2 = compute_weights(train_y_v2_matrix, price=None, normalize=True)

# --- 4. Evaluate V2 models ---
def eval_all(y_true, y_pred, scales, weights):
    return {
        "WRMSSE": wrmsse(y_true, y_pred, scales, weights),
        "MAE":    mae(y_true.ravel(),  y_pred.ravel()),
        "sMAPE":  smape(y_true.ravel(), y_pred.ravel())
    }

c2_results_dict = {
    "LGBM (V2 Tuned)": eval_all(true_matrix_v2, lgb_tuned_matrix, scales_v2, weights_v2),
    "Random Forest (V2)": eval_all(true_matrix_v2, rf_matrix, scales_v2, weights_v2)
}

c2_results = pd.DataFrame(c2_results_dict).T

# --- 5. Combine and Display Final Table ---
final_results_df = pd.concat([c1_results, c2_results])
final_results_df = final_results_df.loc[:, ["WRMSSE", "MAE", "sMAPE"]].sort_values("WRMSSE")

# print("\n--- Final Checkpoint 2 Results ---")
# display(final_results_df.style.format({"WRMSSE":"{:.4f}", "MAE":"{:.4f}", "sMAPE":"{:.2f}%"}))

# Save for your report
final_results_df.to_csv('checkpoint2_results.csv', index=True)
print('Saved final results -> checkpoint2_results.csv')

V2 Matrices built with 1 common series.
Shape of true values: (1, 28)
Shape of LGBM preds:  (1, 28)
Shape of RF preds:    (1, 28)
Saved final results -> checkpoint2_results.csv
